# Correlation Matrix Analysis — Features Dataset

This notebook performs correlation analysis on the engineered features dataset (`Features_dataset.csv`).
As requested, one-hot encoded mill type columns (`mill_Planetary`, `mill_High-Energy`, `mill_Multiple`, `mill_Attritor`) and non-numeric identifiers are excluded to focus on continuous physical parameters and mechanical targets (`hardness_hv`, `strength_hv`).

In [ ]:
# 1. Imports and setup
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# Set project paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'Data/Processed/Features_dataset.csv'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
display(df.head(5))

In [ ]:
# 2. Select numerical features (Ignoring one-hot encoded mill columns)
# Excluded: sample_name, mill_type, mill_Planetary, mill_High-Energy, mill_Multiple, mill_Attritor
NUMERIC_FEATURES = [
    'rpm', 'time_h', 'bpr', 'impact', 'log_time', 'intensity', 'milling_rate'
]
TARGETS = ['hardness_hv', 'strength_hv']
ANALYSIS_COLS = NUMERIC_FEATURES + TARGETS

df_analysis = df[ANALYSIS_COLS]
print(f'Selected {len(ANALYSIS_COLS)} continuous columns for correlation matrix analysis:')
print(ANALYSIS_COLS)

In [ ]:
# 3. Compute Pearson (Linear) and Spearman (Rank) Correlation Matrices
corr_pearson = df_analysis.corr(method='pearson')
corr_spearman = df_analysis.corr(method='spearman')

# P-values for Pearson correlation
p_values_pearson = pd.DataFrame(np.ones_like(corr_pearson), columns=ANALYSIS_COLS, index=ANALYSIS_COLS)
for r in ANALYSIS_COLS:
    for c in ANALYSIS_COLS:
        if r != c:
            _, p = stats.pearsonr(df_analysis[r], df_analysis[c])
            p_values_pearson.loc[r, c] = p

print('=== Pearson Correlation Matrix (Linear) ===')
display(corr_pearson.round(3))

print('=== Spearman Rank Correlation Matrix (Monotonic) ===')
display(corr_spearman.round(3))

In [ ]:
# 4. Correlation with Target Mechanical Properties (Hardness & Strength)
target_corr = pd.DataFrame({
    'Pearson r': corr_pearson.loc[NUMERIC_FEATURES, 'hardness_hv'],
    'Pearson p-value': p_values_pearson.loc[NUMERIC_FEATURES, 'hardness_hv'],
    'Spearman rho': corr_spearman.loc[NUMERIC_FEATURES, 'hardness_hv']
}).sort_values(by='Spearman rho', ascending=False)

print('Feature Correlation with Targets (hardness_hv & strength_hv):')
display(target_corr.round(4))

In [ ]:
# 5. Publication-Quality Visualization (Heatmaps)
sns.set_theme(style='white', font_scale=1.0)
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)

sns.heatmap(corr_pearson, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5,
            cbar_kws={'shrink': .8, 'label': 'Pearson r'}, ax=axes[0])
axes[0].set_title('Pearson Correlation Matrix (Linear)\n[Excluding One-Hot Mill Columns]', fontsize=13, fontweight='bold', pad=12)

sns.heatmap(corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5,
            cbar_kws={'shrink': .8, 'label': 'Spearman ρ'}, ax=axes[1])
axes[1].set_title('Spearman Rank Correlation Matrix (Monotonic)\n[Excluding One-Hot Mill Columns]', fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
heatmap_path = REPORTS_DIR / 'correlation_matrix_heatmap.png'
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Heatmap saved to: {heatmap_path}')

In [ ]:
# 6. Bar Plot of Feature Correlation with Targets
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(target_corr))
width = 0.35

ax.bar(x - width/2, target_corr['Pearson r'], width, label='Pearson r (Linear)', color='#4c72b0')
ax.bar(x + width/2, target_corr['Spearman rho'], width, label='Spearman ρ (Monotonic)', color='#c44e52')

ax.set_ylabel('Correlation Coefficient', fontsize=12, fontweight='bold')
ax.set_title('Feature Correlation with Hardness / Strength (Hv)', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(target_corr.index, rotation=25, ha='right', fontsize=11)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylim(-1, 1)
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', linestyle=':', alpha=0.6)

for i in range(len(target_corr)):
    p_val = target_corr['Pearson r'].iloc[i]
    s_val = target_corr['Spearman rho'].iloc[i]
    ax.text(x[i] - width/2, p_val + (0.04 if p_val >= 0 else -0.08), f'{p_val:.2f}', ha='center', fontsize=9)
    ax.text(x[i] + width/2, s_val + (0.04 if s_val >= 0 else -0.08), f'{s_val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
barplot_path = REPORTS_DIR / 'feature_target_correlation.png'
plt.savefig(barplot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Bar plot saved to: {barplot_path}')